# PURITY Inference

Runs the unified config-driven inference pipeline for PURITY.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pyarrow.parquet as pq

from pioneerml.integration.zenml import load_step_output
from pioneerml.integration.zenml import utils as zenml_utils

PROJECT_ROOT = zenml_utils.setup_repo_pythonpath(Path(zenml_utils.find_project_root()).resolve())
from pioneerml_purity_plugin.purity.pipeline import inference_pipeline, load_config

zenml_utils.setup_zenml_for_notebook(root_path=PROJECT_ROOT, use_in_memory=True)
sys.path.insert(0, str(PROJECT_ROOT / 'artifacts'))
from generate_purity_dummy_parquet import generate_dummy_purity_parquet


## Build Input + Resolve Model

In [ ]:
input_parquet = PROJECT_ROOT / 'artifacts' / 'purity_notebook_inference.parquet'
generate_dummy_purity_parquet(output_path=input_parquet, num_events=32, seed=23)

model_path_file = PROJECT_ROOT / 'artifacts' / 'purity_small_torchscript_path.txt'
if model_path_file.exists():
    model_path = Path(model_path_file.read_text(encoding='utf-8').strip()).resolve()
else:
    candidates = sorted((PROJECT_ROOT / 'artifacts' / 'purity_notebook_export').glob('*_torchscript.pt'))
    if not candidates:
        raise FileNotFoundError('Run training notebook first to produce a torchscript model.')
    model_path = candidates[-1].resolve()

model_path

## Patch Config and Run

In [ ]:
cfg = load_config()['inference']
cfg['model_handle_builder']['model_handle']['config']['model_path'] = str(model_path)
cfg['inference']['loader_manager']['config']['input_sources_spec']['main_sources'] = [str(input_parquet)]
cfg['inference']['loader_manager']['config']['input_sources_spec']['optional_sources_by_name'] = {}
cfg['inference']['loader_manager']['config']['input_sources_spec']['source_type'] = 'file'
cfg['inference']['writer']['config']['output_dir'] = str(PROJECT_ROOT / 'artifacts' / 'purity_notebook_predictions')
cfg['inference']['writer']['config']['fallback_output_dir'] = str(PROJECT_ROOT / 'artifacts' / 'purity_notebook_predictions')
cfg['inference']['writer']['config']['write_timestamped'] = False

run = inference_pipeline.with_options(enable_cache=False)(pipeline_config=cfg)
out = load_step_output(run, 'run_inference')
out

In [ ]:
pred_path = Path(load_step_output(run, 'run_inference')['predictions_path'])
tbl = pq.read_table(pred_path)
print(pred_path)
print(tbl.schema)
tbl.slice(0, 3).to_pydict()